In [30]:
# Basic Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# Model Selection
from sklearn.model_selection import RandomizedSearchCV

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

In [31]:
X_train = pd.read_csv(r"D:\Desktop\mlops\customer-churn-monitoring-mlops\notebooks/data/processed/X_train.csv")
X_test = pd.read_csv(r"D:\Desktop\mlops\customer-churn-monitoring-mlops\notebooks/data/processed/X_test.csv")

y_train = pd.read_csv(r"D:\Desktop\mlops\customer-churn-monitoring-mlops\notebooks/data/processed/y_train.csv").squeeze()
y_test = pd.read_csv(r"D:\Desktop\mlops\customer-churn-monitoring-mlops\notebooks/data/processed/y_test.csv").squeeze()

In [32]:
y_test = y_test.to_numpy().ravel()

In [33]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [34]:
type(y_test)

numpy.ndarray

In [35]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8136363636363636
              precision    recall  f1-score   support

           0       0.84      0.92      0.88      1616
           1       0.70      0.52      0.60       584

    accuracy                           0.81      2200
   macro avg       0.77      0.72      0.74      2200
weighted avg       0.80      0.81      0.80      2200



In [46]:
def evaluate_model(true, predicted, probabilities=None):
    accuracy = accuracy_score(true, predicted)
    precision = precision_score(true, predicted)
    recall = recall_score(true, predicted)
    f1 = f1_score(true, predicted)

    if probabilities is not None:
        roc_auc = roc_auc_score(true, probabilities)
        return accuracy, precision, recall, f1, roc_auc

    return accuracy, precision, recall, f1

In [47]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "K-Neighbors Classifier": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=0
    ),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

model_list = []
accuracy_list = []

for name, model in models.items():

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Probability scores for ROC-AUC
    if hasattr(model, "predict_proba"):
        y_test_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_test_prob = None

    train_metrics = evaluate_model(y_train, y_train_pred)
    test_metrics = evaluate_model(y_test, y_test_pred, y_test_prob)

    print(name)
    model_list.append(name)

    print("Training Performance")
    print(f"Accuracy : {train_metrics[0]:.4f}")
    print(f"Precision: {train_metrics[1]:.4f}")
    print(f"Recall   : {train_metrics[2]:.4f}")
    print(f"F1 Score : {train_metrics[3]:.4f}")

    print("----------------------------------")

    print("Test Performance")
    print(f"Accuracy : {test_metrics[0]:.4f}")
    print(f"Precision: {test_metrics[1]:.4f}")
    print(f"Recall   : {test_metrics[2]:.4f}")
    print(f"F1 Score : {test_metrics[3]:.4f}")

    print("\nClassification Report")
    print(classification_report(y_test, y_test_pred))

    if len(test_metrics) == 5:
        print(f"ROC-AUC  : {test_metrics[4]:.4f}")

    accuracy_list.append(test_metrics[0])

    print("=" * 40)
    print()

Logistic Regression
Training Performance
Accuracy : 0.8381
Precision: 0.7317
Recall   : 0.5913
F1 Score : 0.6540
----------------------------------
Test Performance
Accuracy : 0.8218
Precision: 0.7051
Recall   : 0.5651
F1 Score : 0.6274

Classification Report
              precision    recall  f1-score   support

           0       0.85      0.91      0.88      1616
           1       0.71      0.57      0.63       584

    accuracy                           0.82      2200
   macro avg       0.78      0.74      0.76      2200
weighted avg       0.81      0.82      0.82      2200

ROC-AUC  : 0.8782

K-Neighbors Classifier
Training Performance
Accuracy : 0.8605
Precision: 0.7594
Recall   : 0.6747
F1 Score : 0.7146
----------------------------------
Test Performance
Accuracy : 0.7900
Precision: 0.6230
Recall   : 0.5291
F1 Score : 0.5722

Classification Report
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1616
           1       0.6

In [48]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

,Model Name,R2_Score
4,XGBoost,0.369321
1,K-Neighbors Classifier,0.335506
5,LightGBM,0.280153
2,Decision Tree,0.219441
0,Logistic Regression,-0.000223
3,Random Forest,-0.291452


AdaBoost has:

✅ Highest Accuracy (82.82%)

✅ Highest Precision (71.28%)

✅ Highest Recall (59.08%)

✅ Highest F1-score (64.61%)

ROC-AUC (87.46%), which is only 0.0036 lower than Logistic Regression's 87.82%—a very small difference.
